In [3]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Ipsos for NIESR pre-Budget poll, fieldwork 9-13 Oct 2026, GB adults aged 18-75 (n=1,080)
# QB13: how likely is it that financial markets will severely lose confidence in the
# UK government's ability to pay back its debt over the next five years?
# Table 29, % of each group.
# Unweighted bases: all adults 1,080; Reform UK 123; Conservative 155; Lib Dem 86; Labour 319
# (* = small base, flagged by Ipsos)
df = pd.DataFrame({
    "group": ["All adults", "Reform UK*", "Conservative", "Liberal Democrat*", "Labour"],
    "Very likely": [16, 39, 19, 4, 10],
    "Fairly likely": [36, 32, 51, 44, 30],
})
df["total"] = df["Very likely"] + df["Fairly likely"]
df["total_label"] = df["total"].astype(str) + "%"

order = df["group"].tolist()
answers = ["Very likely", "Fairly likely"]
colors = ["#2563eb", "#94a3b8"]          # canonical primary / secondary
ink = "#0f172a"                          # dark text for labels on the light segment

seg = df.melt(id_vars="group", value_vars=answers, var_name="answer", value_name="share")
seg["rank"] = seg["answer"].map({"Very likely": 0, "Fairly likely": 1})
seg = seg.sort_values(["group", "rank"])
seg["end"] = seg.groupby("group")["share"].cumsum()
seg["start"] = seg["end"] - seg["share"]
seg["mid"] = (seg["start"] + seg["end"]) / 2
seg["label"] = seg["share"].astype(str)

# seriesHover: hover a segment and the other series fade
hover = alt.selection_point(fields=["answer"], on="pointerover", clear="pointerout")
fade = alt.condition(hover, alt.value(1), alt.value(0.35))

y = alt.Y("group:N", sort=order, title=None,
          axis=alt.Axis(labelLimit=200, grid=True))      # dotted guide lines (gridDash set in config)

# horizontal floating legend (orient none), sitting between the subtitle and the plot
legend = alt.Legend(title=None, orient="none", direction="horizontal",
                    legendX=0, legendY=-30, symbolType="square", symbolSize=110,
                    labelFontSize=11, columnPadding=16)

bars = alt.Chart(seg).mark_bar(cornerRadius=2, size=26).encode(
    y=y,
    x=alt.X("start:Q", title="% of group", scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(values=[0, 25, 50, 75, 100], grid=False)),   # no x-grid
    x2="end:Q",
    color=alt.Color("answer:N", sort=answers,
                    scale=alt.Scale(domain=answers, range=colors), legend=legend),
    opacity=fade,
    tooltip=[alt.Tooltip("group:N", title="Group"),
             alt.Tooltip("answer:N", title="Answer"),
             alt.Tooltip("share:Q", title="% of group")],
).add_params(hover)

lab_very = alt.Chart(seg[(seg["answer"] == "Very likely") & (seg["share"] >= 8)]).mark_text(
    color="white", fontSize=11).encode(y=y, x="mid:Q", text="label:N", opacity=fade)
lab_fair = alt.Chart(seg[seg["answer"] == "Fairly likely"]).mark_text(
    color=ink, fontSize=11).encode(y=y, x="mid:Q", text="label:N", opacity=fade)

totals = alt.Chart(df).mark_text(align="left", dx=6, fontSize=12, fontWeight="bold",
                                 color=ink).encode(y=y, x="total:Q", text="total_label:N")

title = alt.Title(
    "Expecting markets to lose confidence in UK debt, by 2024 vote",
    subtitle="Share saying it is very or fairly likely over the next five years, %",
    offset=38)                                           # leaves room for the floating legend

chart = (
    (bars + lab_very + lab_fair + totals)
    .properties(width=560, height=250, title=title, padding={"right": 20, "bottom": 10})
    .configure(background="white", font="Instrument Sans, Helvetica, Arial, sans-serif")
    .configure_axis(gridDash=[1, 5])
    .configure_axisX(grid=False)
    .configure_axisY(titleAngle=0)
    .configure_view(stroke="transparent")
)

styles.save(chart, path=".", name="markets_confidence_by_vote", svg=True)
chart

alt.LayerChart(...)

In [27]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Ipsos for NIESR pre-Budget poll, fieldwork 9-13 Oct 2026, GB adults aged 18-75 (n=1,080)
# QB13, Table 29, % of each 2024 general election vote group saying very or fairly likely.
raw = pd.DataFrame({
    "group":  ["All adults", "Reform UK", "Conservative", "Liberal Democrat", "Labour"],
    "very":   [16, 39, 19, 4, 10],
    "fairly": [36, 32, 51, 44, 30],
})
raw["likely"] = raw["very"] + raw["fairly"]
raw["zero"] = 0

party_colours = {
    "All adults":       "#676A86",   # grey
    "Reform UK":        "#1EB8D0",   # turquoise
    "Conservative":     "#0087DC",   # blue
    "Liberal Democrat": "#FF6400",   # orange
    "Labour":           "#E41C3E",   # red
}


def tint(hex_colour, amount):
    """Mix a hex colour with white (0 = unchanged, 1 = white)."""
    h = hex_colour.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return "#{:02X}{:02X}{:02X}".format(*(round(c + (255 - c) * amount) for c in (r, g, b)))


raw["solid"] = raw["group"].map(party_colours)
raw["light"] = raw["solid"].map(lambda c: tint(c, 0.6))

# rank by the headline number: All adults sits in the ranking, so above/below it reads as above/below average
order = raw.sort_values("likely", ascending=False)["group"].tolist()

y = alt.Y("group:N", sort=order, title=None, axis=alt.Axis(domain=False, ticks=False, labelLimit=220))
x = alt.X("zero:Q", title=None, scale=alt.Scale(domain=[0, 100]),
          axis=alt.Axis(values=[0, 25, 50, 75, 100], labelExpr="datum.value + '%'"))
tip = [alt.Tooltip("group:N", title="Group"),
       alt.Tooltip("very:Q", title="Very likely, %"),
       alt.Tooltip("fairly:Q", title="Fairly likely, %"),
       alt.Tooltip("likely:Q", title="Total likely, %")]

# one full-length bar (very + fairly, light) with the very-likely part drawn over it (solid):
# square ends and no stacked segments, so there are no seams or rounded joins
total_bar = alt.Chart(raw).mark_bar(size=32).encode(
    y=y, x=x, x2="likely:Q", tooltip=tip, color=alt.Color("light:N", scale=None, legend=None))
very_bar = alt.Chart(raw).mark_bar(size=32).encode(
    y=y, x=x, x2="very:Q", tooltip=tip, color=alt.Color("solid:N", scale=None, legend=None))

title = alt.Title(
    "Expecting markets to lose confidence in UK debt, by 2024 vote",
    subtitle="Share saying it is very (darker) or fairly likely over the next five years, %")

chart = (
    (total_bar + very_bar)
    .properties(width=480, height=250, title=title, padding={"right": 20, "bottom": 10, "top": 8})
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor="#676A86", titleColor="#676A86")
)

styles.save(chart, path=".", name="markets_confidence_by_vote", svg=True)
chart.save("markets_confidence_by_vote.png", scale_factor=2.0)
chart

alt.LayerChart(...)

In [28]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Ipsos for NIESR pre-Budget poll, fieldwork 9-13 Oct 2026, GB adults aged 18-75 (n=1,080)
# QB13, Table 29, % of each 2024 general election vote group saying very or fairly likely.
raw = pd.DataFrame({
    "group":  ["All adults", "Reform UK", "Conservative", "Liberal Democrat", "Labour"],
    "very":   [16, 39, 19, 4, 10],
    "fairly": [36, 32, 51, 44, 30],
})
raw["likely"] = raw["very"] + raw["fairly"]
raw["zero"] = 0
raw["label"] = raw["likely"].astype(str) + "%"

party_colours = {
    "All adults":       "#676A86",   # grey
    "Reform UK":        "#1EB8D0",   # turquoise
    "Conservative":     "#0087DC",   # blue
    "Liberal Democrat": "#FF6400",   # orange
    "Labour":           "#E41C3E",   # red
}


def tint(hex_colour, amount):
    """Mix a hex colour with white (0 = unchanged, 1 = white)."""
    h = hex_colour.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return "#{:02X}{:02X}{:02X}".format(*(round(c + (255 - c) * amount) for c in (r, g, b)))


raw["solid"] = raw["group"].map(party_colours)
raw["light"] = raw["solid"].map(lambda c: tint(c, 0.6))

# rank by the headline number: All adults sits in the ranking, so above/below it reads as above/below average
order = raw.sort_values("likely", ascending=False)["group"].tolist()

y = alt.Y("group:N", sort=order, title=None, axis=alt.Axis(domain=False, ticks=False, labelLimit=220))
x = alt.X("zero:Q", scale=alt.Scale(domain=[0, 100]), axis=None)
tip = [alt.Tooltip("group:N", title="Group"),
       alt.Tooltip("very:Q", title="Very likely, %"),
       alt.Tooltip("fairly:Q", title="Fairly likely, %"),
       alt.Tooltip("likely:Q", title="Total likely, %")]

# one full-length bar (very + fairly, light) with the very-likely part drawn over it (solid):
# square ends and no stacked segments, so there are no seams or rounded joins
total_bar = alt.Chart(raw).mark_bar(size=32).encode(
    y=y, x=x, x2="likely:Q", tooltip=tip, color=alt.Color("light:N", scale=None, legend=None))
very_bar = alt.Chart(raw).mark_bar(size=32).encode(
    y=y, x=x, x2="very:Q", tooltip=tip, color=alt.Color("solid:N", scale=None, legend=None))

labels = alt.Chart(raw).mark_text(align="left", dx=8, fontSize=14, fontWeight="bold").encode(
    y=y, x="likely:Q", text="label:N")

title = alt.Title(
    "Expecting markets to lose confidence in UK debt, by 2024 vote",
    subtitle="Share saying it is very (darker) or fairly likely over the next five years, %")

chart = (
    (total_bar + very_bar + labels)
    .properties(width=480, height=250, title=title, padding={"right": 20, "bottom": 10, "top": 8})
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor="#676A86", titleColor="#676A86")
)

styles.save(chart, path=".", name="markets_confidence_by_vote", svg=True)
chart.save("markets_confidence_by_vote.png", scale_factor=2.0)
chart

alt.LayerChart(...)

In [32]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Ipsos for NIESR pre-Budget poll, fieldwork 9-13 Oct 2026, GB adults aged 18-75 (n=1,080)
# QB13, Table 29, % of each 2024 general election vote group saying very or fairly likely.
raw = pd.DataFrame({
    "group":  ["All adults", "Reform UK", "Conservative", "Liberal Democrat", "Labour"],
    "very":   [16, 39, 19, 4, 10],
    "fairly": [36, 32, 51, 44, 30],
})
raw["likely"] = raw["very"] + raw["fairly"]
raw["zero"] = 0
raw["label"] = raw["likely"].astype(str) + "%"

party_colours = {
    "All adults":       "#676A86",   # grey
    "Reform UK":        "#1EB8D0",   # turquoise
    "Conservative":     "#0087DC",   # blue
    "Liberal Democrat": "#FF6400",   # orange
    "Labour":           "#E41C3E",   # red
}


def tint(hex_colour, amount):
    """Mix a hex colour with white (0 = unchanged, 1 = white)."""
    h = hex_colour.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return "#{:02X}{:02X}{:02X}".format(*(round(c + (255 - c) * amount) for c in (r, g, b)))


raw["solid"] = raw["group"].map(party_colours)
raw["light"] = raw["solid"].map(lambda c: tint(c, 0.6))

# rank by the headline number: All adults sits in the ranking, so above/below it reads as above/below average
order = raw.sort_values("likely", ascending=False)["group"].tolist()

y = alt.Y("group:N", sort=order, title=None, axis=alt.Axis(domain=False, ticks=False, labelLimit=220))
x = alt.X("zero:Q", scale=alt.Scale(domain=[0, 100]), axis=None)
tip = [alt.Tooltip("group:N", title="Group"),
       alt.Tooltip("very:Q", title="Very likely, %"),
       alt.Tooltip("fairly:Q", title="Fairly likely, %"),
       alt.Tooltip("likely:Q", title="Total likely, %")]

# one full-length bar (very + fairly, light) with the very-likely part drawn over it (solid):
# square ends and no stacked segments, so there are no seams or rounded joins
total_bar = alt.Chart(raw).mark_bar(size=32).encode(
    y=y, x=x, x2="likely:Q", tooltip=tip, color=alt.Color("light:N", scale=None, legend=None))
very_bar = alt.Chart(raw).mark_bar(size=32).encode(
    y=y, x=x, x2="very:Q", tooltip=tip, color=alt.Color("solid:N", scale=None, legend=None))

labels = alt.Chart(raw).mark_text(align="left", dx=8, fontSize=14, fontWeight="bold").encode(
    y=y, x="likely:Q", text="label:N")

chart = (
    (total_bar + very_bar + labels)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor="#676A86", titleColor="#676A86")
)

styles.save(chart, path=".", name="markets_confidence_by_vote", svg=True)
chart.save("markets_confidence_by_vote.png", scale_factor=2.0)
chart

alt.LayerChart(...)